In [5]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

from utils import alert_training_complete
from utils.dqn_utils import DQN, ReplayBuffer, save_dqn_checkpoint

In [2]:
env = gym.make("LunarLander-v3")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("State size:", state_size)
print("Action size:", action_size)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

State size: 8
Action size: 4
Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space: Discrete(4)


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

Using: cpu


In [6]:
num_episodes = 1500

gamma = 0.99
learning_rate = 5e-4

batch_size = 64

buffer_capacity = 100_000
min_buffer_size = 1_000

epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay_steps = 100_000

target_update_frequency = 1_000

In [7]:
policy_net = DQN(
    state_size,
    action_size
).to(device)

target_net = DQN(
    state_size,
    action_size
).to(device)

target_net.load_state_dict(
    policy_net.state_dict()
)

target_net.eval()

DQN(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)

In [8]:
replay_buffer = ReplayBuffer(
    buffer_capacity
)

optimizer = optim.Adam(
    policy_net.parameters(),
    lr=learning_rate
)

loss_function = nn.SmoothL1Loss()

In [9]:
def get_epsilon(step):

    fraction = min(
        step / epsilon_decay_steps,
        1.0
    )

    epsilon = (
        epsilon_start
        + fraction
        * (epsilon_end - epsilon_start)
    )

    return epsilon

In [10]:
def choose_action(state, epsilon):

    # Exploration
    if np.random.random() < epsilon:
        return env.action_space.sample()

    # Exploitation
    state_tensor = torch.tensor(
        state,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        q_values = policy_net(
            state_tensor
        )

    return q_values.argmax(dim=1).item()

In [11]:
def train_dqn():

    if len(replay_buffer) < min_buffer_size:
        return

    (
        states,
        actions,
        rewards,
        next_states,
        terminated
    ) = replay_buffer.sample(
        batch_size,
        device
    )

    # Move batch to GPU/CPU
    states = states.to(device)
    actions = actions.to(device)
    rewards = rewards.to(device)
    next_states = next_states.to(device)
    terminated = terminated.to(device)

    # ------------------------------------------------
    # Current Q(s, a)
    # ------------------------------------------------

    current_q = policy_net(
        states
    ).gather(
        1,
        actions.unsqueeze(1)
    ).squeeze(1)

    # ------------------------------------------------
    # Bellman target
    # ------------------------------------------------

    with torch.no_grad():

        next_q = target_net(
            next_states
        ).max(dim=1).values

        target_q = (
            rewards
            + gamma
            * next_q
            * (1 - terminated)
        )

    # ------------------------------------------------
    # Update policy network
    # ------------------------------------------------

    loss = loss_function(
        current_q,
        target_q
    )

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        policy_net.parameters(),
        10.0
    )

    optimizer.step()

In [ ]:
episode_rewards = []
average_rewards = []

total_steps = 0

for episode in range(num_episodes):

    state, info = env.reset()

    terminated = False
    truncated = False

    episode_reward = 0

    while not (terminated or truncated):

        # --------------------------------------------
        # Choose action
        # --------------------------------------------

        epsilon = get_epsilon(
            total_steps
        )

        action = choose_action(
            state,
            epsilon
        )

        # --------------------------------------------
        # Apply action
        # --------------------------------------------

        (
            next_state,
            reward,
            terminated,
            truncated,
            info
        ) = env.step(action)

        # --------------------------------------------
        # Store experience
        # --------------------------------------------

        replay_buffer.push(
            state,
            action,
            reward,
            next_state,
            terminated
        )

        # --------------------------------------------
        # Train network
        # --------------------------------------------

        train_dqn()

        # --------------------------------------------
        # Update state
        # --------------------------------------------

        state = next_state

        episode_reward += reward
        total_steps += 1

        # --------------------------------------------
        # Update target network
        # --------------------------------------------

        if (
            total_steps
            % target_update_frequency
            == 0
        ):

            target_net.load_state_dict(
                policy_net.state_dict()
            )

    # ------------------------------------------------
    # Episode statistics
    # ------------------------------------------------

    episode_rewards.append(
        episode_reward
    )

    average_reward = np.mean(
        episode_rewards[-100:]
    )

    average_rewards.append(
        average_reward
    )

    # ------------------------------------------------
    # Print progress
    # ------------------------------------------------

    if episode % 25 == 0:

        print(
            f"Episode: {episode:4d} | "
            f"Reward: {episode_reward:8.2f} | "
            f"Average(100): {average_reward:8.2f} | "
            f"Epsilon: {epsilon:.3f}"
        )

    # ------------------------------------------------
    # Solved
    # ------------------------------------------------

    if (
        len(episode_rewards) >= 100
        and average_reward >= 200
    ):

        print(
            f"\nEnvironment solved "
            f"at episode {episode}!"
        )

        break

env.close()

In [ ]:
# Notify the user only after the complete training loop has finished.
alert_training_complete(
    "Training finished - the LunarLander DQN is ready."
)

In [ ]:
from pathlib import Path

model_path = Path("models") / "lunarlander_dqn.pth"
model_path.parent.mkdir(parents=True, exist_ok=True)

save_dqn_checkpoint(
    model_path,
    policy_net,
    target_net=target_net,
    optimizer=optimizer,
    metadata={
        "episodes": len(episode_rewards),
        "gamma": gamma,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epsilon_start": epsilon_start,
        "epsilon_end": epsilon_end,
        "epsilon_decay_steps": epsilon_decay_steps,
        "target_update_frequency": target_update_frequency,
        "hidden_sizes": (128, 128),
        "environment": "LunarLander-v3",
        "total_steps": total_steps,
    },
)

print(f"Saved trained DQN model to: {model_path}")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    episode_rewards,
    alpha=0.35,
    label="Episode reward"
)

plt.plot(
    average_rewards,
    label="100-episode average"
)

plt.axhline(
    200,
    linestyle="--",
    label="Solved threshold"
)

plt.xlabel("Episode")
plt.ylabel("Reward")

plt.title(
    "LunarLander DQN Training"
)

plt.legend()
plt.grid()

plt.show()

In [ ]:
test_env = gym.make(
    "LunarLander-v3",
    render_mode="human"
)

num_test_episodes = 10

for episode in range(num_test_episodes):

    state, info = test_env.reset()

    terminated = False
    truncated = False

    total_reward = 0

    while not (terminated or truncated):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        with torch.no_grad():

            q_values = policy_net(
                state_tensor
            )

            action = q_values.argmax(
                dim=1
            ).item()

        (
            state,
            reward,
            terminated,
            truncated,
            info
        ) = test_env.step(action)

        total_reward += reward

    print(
        f"Test episode {episode + 1}: "
        f"{total_reward:.2f}"
    )

test_env.close()